# 05 - Líneas base

Antes de entrenar un modelo hay que saber **contra qué se compara** (Semana 2: "comparar contra una línea base
trivial"). Este notebook evalúa reglas simples que no aprenden nada, o casi nada, y fija el número a superar.

| Línea base | Predicción para la línea L, el día D, la hora H | ¿Se ajusta con train? |
|---|---|---|
| `media_train` | El promedio de todas las filas de train | Sí (un número) |
| `perfil_historico` | El promedio de train para (L, H, tipo de día de D) | Sí (una tabla) |
| `ayer` | Los pasajeros de L a la hora H del día D-1 | No |
| `semana_pasada` | Los pasajeros de L a la hora H del día D-7 | No |
| `prom_4sem` | El promedio de D-7, D-14, D-21 y D-28 a la hora H | No |
| `mediana_4sem` | La mediana de esos mismos 4 valores | No |

**Referencia: `mediana_4sem`.** En el EDA fue la variable más correlacionada con el objetivo (0,972). Todo modelo
de `06` se juzga por cuánto mejora sobre ella.

### Métricas (Semana 3)

| Métrica | Fórmula | Lectura |
|---|---|---|
| MAE | $\frac{1}{N}\sum \lvert \hat y - y \rvert$ | Error típico en pasajeros por hora |
| RMSE | $\sqrt{\frac{1}{N}\sum (\hat y - y)^2}$ | Como el MAE, pero castiga más los errores grandes. Siempre MAE ≤ RMSE |
| R² | $1 - \frac{\sum (\hat y - y)^2}{\sum (y - \bar y)^2}$ | Cuánto mejora sobre predecir siempre la media. 0 = igual que la media |
| WAPE | $\frac{\sum \lvert \hat y - y \rvert}{\sum y}$ | Error total como fracción de los pasajeros totales |
| Sesgo | $\frac{1}{N}\sum (\hat y - y)$ | Hacia qué lado se equivoca: > 0 se pasa, < 0 se queda corto |
| Mejora | $1 - \frac{\text{MAE}}{\text{MAE}_{\text{mediana 4 sem}}}$ | 0,10 = 10 % menos error que la referencia; negativo = peor |

Los intervalos del 95 % se calculan por **bootstrap remuestreando días** (no filas): las filas de un mismo día
comparten calendario y clima, así que no son independientes.

**Todo se evalúa en validación** (octubre a diciembre de 2024), solo con filas completas. Test no se toca.

In [0]:
%run ./00_setup

In [0]:
import importlib
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from bondis import metricas, variables

importlib.reload(metricas)
importlib.reload(variables)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)

REFERENCIA = "mediana_4sem"

gold = variables.con_tipo_dia(spark.table(config.TABLE_GOLD)).where(F.col("fila_completa"))
train = gold.where(F.col("particion") == "train")
val = gold.where(F.col("particion") == "val")

r = val.agg(F.count("*").alias("filas"), F.countDistinct("fecha").alias("dias"),
            F.min("fecha").alias("desde"), F.max("fecha").alias("hasta")).first()
print(f"Validación: {r['filas']:,} filas, {r['dias']} días ({r['desde']} a {r['hasta']})")

## 1. Ajuste de las líneas base que usan train

- `media_train`: un solo número.
- `perfil_historico`: promedio por línea, hora y tipo de día. Si una combinación no aparece en train (por ejemplo,
  una línea que nunca circuló un feriado a esa hora), se usa el promedio por línea y hora.

In [0]:
media_train = train.agg(F.avg("pasajeros")).first()[0]
perfil = train.groupBy("linea", "hora", "tipo_dia").agg(F.avg("pasajeros").alias("_perfil"))
perfil_lh = train.groupBy("linea", "hora").agg(F.avg("pasajeros").alias("_perfil_lh"))
tamanos = variables.tamano_linea(train)

val_pred = (
    val.join(perfil, ["linea", "hora", "tipo_dia"], "left")
    .join(perfil_lh, ["linea", "hora"], "left")
    .join(tamanos, "linea", "left")
    .withColumn("pred_perfil", F.coalesce("_perfil", "_perfil_lh"))
    .withColumn("pred_media", F.lit(media_train))
    .fillna({"tamano_linea": "chica"})
)

PREDICCIONES = {
    "media_train": "pred_media",
    "perfil_historico": "pred_perfil",
    "ayer": "pasajeros_lag_1d",
    "semana_pasada": "pasajeros_lag_7d",
    "prom_4sem": "pasajeros_prom_4sem",
    "mediana_4sem": "pasajeros_mediana_4sem",
}

print(f"media_train = {media_train:.1f} pasajeros por hora")
sin_perfil = val_pred.where(F.col("_perfil").isNull()).count()
print(f"Filas de val sin combinación (línea, hora, tipo de día) en train: {sin_perfil:,}")

## 2. Resultados globales en validación

Una sola pasada de Spark calcula las sumas por día para las seis líneas base. Todo lo que sigue sale de esas sumas.

In [0]:
est = metricas.estadisticos_por_dia(val_pred, "pasajeros", PREDICCIONES, grupos=["tipo_dia", "tamano_linea"])

globales = metricas.agregar_mejora(metricas.metricas(est), REFERENCIA)
intervalos = metricas.bootstrap_dias(est, REFERENCIA, n_rep=1000, seed=config.SEED)
globales = globales.merge(intervalos, on="modelo")
orden = list(PREDICCIONES)
globales = globales.set_index("modelo").loc[orden].reset_index()


def fmt_intervalo(v, inf, sup, dec=1, pct=False):
    f = (lambda x: f"{100 * x:.{dec}f} %") if pct else (lambda x: f"{x:,.{dec}f}")
    return f"{f(v)} [{f(inf)}; {f(sup)}]"


vista = pd.DataFrame({
    "modelo": globales["modelo"],
    "MAE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["mae", "mae_inf", "mae_sup"]].to_numpy()],
    "RMSE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["rmse", "rmse_inf", "rmse_sup"]].to_numpy()],
    "R²": globales["r2"].round(3),
    "WAPE": (100 * globales["wape"]).round(1).astype(str) + " %",
    "sesgo": globales["sesgo"].round(1),
    "mejora [IC 95 %]": [fmt_intervalo(*x, pct=True) for x in globales[["mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]].to_numpy()],
})
print(vista.to_string(index=False))

**Cómo leer la tabla.**
- El R² de `media_train` debería dar cerca de 0 (o negativo, porque la media es la de train y no la de val).
  Si las demás líneas base tienen R² cercano a 1, se confirma lo que anticipó el EDA: el R² no sirve para
  distinguir modelos en este problema, y la columna que importa es la **mejora** sobre la referencia.
- Si el intervalo de la mejora de un modelo no incluye el 0, la diferencia con la referencia es real y no
  efecto de qué días tocaron en validación.

## 3. ¿Dónde falla cada línea base?

### 3.1 Por tipo de día

Lo esperado: las reglas basadas en semanas anteriores fallan en los días que no se parecen a esas semanas
(feriados), y `perfil_historico` falla cuando cambia el nivel general (por ejemplo, fin de clases en diciembre).

In [0]:
por_tipo = metricas.agregar_mejora(metricas.metricas(est, por=["tipo_dia"]), REFERENCIA, por=["tipo_dia"])
dias_tipo = est.groupby("tipo_dia")["fecha"].nunique().rename("días")
tipos = [t for t in variables.TIPOS_DIA if t in dias_tipo.index]

print("MAE por tipo de día (pasajeros por hora):")
t = por_tipo.pivot(index="modelo", columns="tipo_dia", values="mae").loc[orden, tipos].round(1)
print(pd.concat([dias_tipo[tipos].to_frame().T, t]).to_string(), "\n")

print("Sesgo por tipo de día (> 0: se pasa):")
print(por_tipo.pivot(index="modelo", columns="tipo_dia", values="sesgo").loc[orden, tipos].round(1).to_string())

### 3.2 Por tamaño de línea

Las líneas se agrupan por pasajeros en train: las 20 más grandes, las 30 siguientes y el resto. Como el error
crece con el tamaño de la línea (EDA, sección 4), el MAE global lo deciden las líneas grandes; el WAPE permite
comparar grupos de distinto tamaño.

In [0]:
por_tam = metricas.metricas(est, por=["tamano_linea"])
grupos_tam = [g for g in ["grande", "mediana", "chica"] if g in set(por_tam["tamano_linea"])]
for col, titulo in (("mae", "MAE"), ("wape", "WAPE")):
    t = por_tam.pivot(index="modelo", columns="tamano_linea", values=col).loc[orden, grupos_tam]
    print(f"{titulo} por tamaño de línea:")
    print((t * 100).round(1).astype(str).add(" %").to_string() if col == "wape" else t.round(1).to_string(), "\n")

### 3.3 Día por día

MAE de cada día de validación para tres líneas base. Los picos son los días en que la regla se equivoca;
deberían coincidir con días especiales (líneas grises punteadas) o con cambios en el calendario escolar
(líneas rojas: el primer día en que cambia el estado de clases de primaria o de media).

In [0]:
info_dias = (val.select("fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria", "clases_media")
             .distinct().toPandas())
info_dias["fecha"] = pd.to_datetime(info_dias["fecha"])

diario = metricas.metricas(est, por=["fecha"])
fig, ax = plt.subplots()
for m in ["semana_pasada", "mediana_4sem", "perfil_historico"]:
    s = diario[diario["modelo"] == m].sort_values("fecha")
    ax.plot(s["fecha"], s["mae"], marker=".", ms=4, lw=1, label=m)
tope = ax.get_ylim()[1]
for _, d in info_dias.dropna(subset=["nombre_dia_especial"]).iterrows():
    ax.axvline(d["fecha"], color="gray", ls=":", lw=1)
    ax.text(d["fecha"], tope, d["nombre_dia_especial"], rotation=90, va="top", ha="right", fontsize=7)

# Cambios de régimen escolar: días en que clases_primaria o clases_media cambia respecto del día anterior
cal = info_dias.sort_values("fecha").set_index("fecha")[["clases_primaria", "clases_media"]]
for nivel in ("primaria", "media"):
    c = cal[f"clases_{nivel}"].astype(int).diff()
    for f, v in c[c != 0].dropna().items():
        etiqueta = f"{'inicio' if v > 0 else 'fin'} de clases ({nivel})"
        ax.axvline(f, color="tab:red", ls="--", lw=1)
        ax.text(f, tope, etiqueta, rotation=90, va="top", ha="right", fontsize=7, color="tab:red")

ax.set(title="MAE por día en validación", ylabel="MAE (pasajeros por hora)")
ax.legend(loc="center left")
plt.show()

peores = (diario[diario["modelo"] == REFERENCIA].nlargest(10, "mae")
          .merge(info_dias, on="fecha")[["fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria",
                                         "clases_media", "mae", "sesgo"]])
print(f"Los 10 días con más error de {REFERENCIA}:")
print(peores.round(1).to_string(index=False))

## 4. Guardar los resultados

Los resultados globales se guardan en `resultados_modelos` con `etapa = 'baseline'`. `06` agrega los suyos
con otra etapa, así `07` compara todo desde una sola tabla. Re-ejecutar este notebook reemplaza solo las filas
de la etapa `baseline`.

In [0]:
ETAPA = "baseline"
salida = globales.assign(etapa=ETAPA, conjunto="val", referencia=REFERENCIA,
                         fecha_ejecucion=datetime.now().strftime("%Y-%m-%d %H:%M"))
columnas = ["etapa", "conjunto", "modelo", "referencia", "filas", "mae", "mae_inf", "mae_sup", "rmse",
            "rmse_inf", "rmse_sup", "r2", "wape", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup",
            "fecha_ejecucion"]
salida = salida[columnas].astype({c: "float64" for c in columnas if c not in
                                  ("etapa", "conjunto", "modelo", "referencia", "fecha_ejecucion", "filas")})

escritor = spark.createDataFrame(salida).write.format("delta").mode("overwrite").option("mergeSchema", "true")
if spark.catalog.tableExists(config.TABLE_RESULTADOS):
    escritor.option("replaceWhere", f"etapa = '{ETAPA}'").saveAsTable(config.TABLE_RESULTADOS)
else:
    escritor.saveAsTable(config.TABLE_RESULTADOS)

print(spark.table(config.TABLE_RESULTADOS).where(F.col("etapa") == ETAPA)
      .select("modelo", "mae", "rmse", "r2", "mejora_mae").toPandas().round(3).to_string(index=False))

## Conclusiones

Completar con los resultados.

| Pregunta | Resultado |
|---|---|
| ¿Qué línea base tiene menos error? ¿Es la referencia? | |
| MAE de la referencia (el número a superar en `06`) | |
| ¿El R² distingue entre las líneas base? | |
| ¿`perfil_historico` (que sí usa train) le gana a las reglas de semanas anteriores? | |
| ¿En qué tipo de día falla más la referencia? ¿Con qué signo de sesgo? | |
| ¿Los peores días son días especiales? ¿Cuáles? | |
| ¿Qué variables de gold podrían haber evitado esos errores? | |